In [5]:
import os
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer
import torch

In [7]:
DATA_PATH = "data"

train_df = pd.read_csv(os.path.join(DATA_PATH, "train.tsv"), sep="\t")
val_df = pd.read_csv(os.path.join(DATA_PATH, "valid.tsv"), sep="\t")
test_df = pd.read_csv(os.path.join(DATA_PATH, "test.tsv"), sep="\t")

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (10239, 14)
Val: (1283, 14)
Test: (1266, 14)


In [8]:
def map_labels(label):
    if label in ["true", "mostly-true"]:
        return "True"
    elif label in ["half-true", "barely-true"]:
        return "Nuanced"
    elif label in ["false", "pants-fire"]:
        return "False"
    else:
        return None

train_df["label_grouped"] = train_df["label"].apply(map_labels)
val_df["label_grouped"] = val_df["label"].apply(map_labels)
test_df["label_grouped"] = test_df["label"].apply(map_labels)

# supprimer les lignes non mappées
train_df = train_df.dropna(subset=["label_grouped"])
val_df = val_df.dropna(subset=["label_grouped"])
test_df = test_df.dropna(subset=["label_grouped"])

KeyError: 'label'

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)  # enlever espaces multiples
    text = text.strip()
    return text

train_df["text_clean"] = train_df["text"].apply(clean_text)
val_df["text_clean"] = val_df["text"].apply(clean_text)
test_df["text_clean"] = test_df["text"].apply(clean_text)

In [ ]:
label_encoder = LabelEncoder()

train_labels = label_encoder.fit_transform(train_df["label_grouped"])
val_labels = label_encoder.transform(val_df["label_grouped"])
test_labels = label_encoder.transform(test_df["label_grouped"])

print(label_encoder.classes_)

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

train_encodings = tokenize(train_df["text_clean"])
val_encodings = tokenize(val_df["text_clean"])
test_encodings = tokenize(test_df["text_clean"])

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)
test_dataset = Dataset(test_encodings, test_labels)